# Organise Redirects file

In [ ]:
from __future__ import annotations

import csv
import os
import re

import numpy as np
import pandas as pd
from tqdm import tqdm

In [ ]:
main_dir = os.path.dirname(os.getcwd())
metadata_tsv_path = os.path.join(main_dir, "data/inter/wiki_metadata.tsv")
title_info_tsv_path = os.path.join(main_dir, "data/output/wiki_title_info.tsv")

In [ ]:
metadata_df = pd.read_csv(metadata_tsv_path, sep="\t", header=0, quoting=csv.QUOTE_NONE)
print(len(metadata_df))
metadata_df.head()

In [ ]:
page_id_ary = metadata_df["PageID"].values.astype(int)
mat_index_ary = metadata_df["MatrixIndex"].values.astype(int)
pagetitle_ser = metadata_df["PageTitle"].astype(str)
redirect_ser = metadata_df["RedirectLocation"].astype(str)

In [ ]:
pagetitle_argsort = pagetitle_ser.argsort()
sorted_page_id = page_id_ary[pagetitle_argsort]
sorted_page_title_ser = pagetitle_ser[pagetitle_argsort]
sorted_mat_index = mat_index_ary[pagetitle_argsort]
sorted_redirect_ser = redirect_ser[pagetitle_argsort]

In [ ]:
total_len = len(sorted_page_title_ser)

new_page_id = np.full(total_len, -1, dtype=int)
new_mat_index = np.full(total_len, -2, dtype=int)
new_is_redirect = np.full(total_len, False, dtype=bool)
new_page_title = [""] * total_len
new_redirect = [""] * total_len

In [ ]:
def get_sorted_redirect(redirect):
    x = np.searchsorted(sorted_page_title_ser, redirect)
    if x < total_len and sorted_page_title_ser.iloc[x] == redirect:
        return x
    return None

In [ ]:
i_r = 0

for i in tqdm(range(total_len)):
    if sorted_mat_index[i] != -1:
        new_page_id[i] = sorted_page_id[i]
        new_mat_index[i] = sorted_mat_index[i]
        new_page_title[i] = sorted_page_title_ser.iloc[i]
        new_is_redirect[i] = False
        new_redirect[i] = sorted_page_title_ser.iloc[i]
    else:
        count_redirect = 0
        i_r = i
        while count_redirect < 10:
            count_redirect += 1
            i_r = get_sorted_redirect(sorted_redirect_ser.iloc[i_r])
            if i_r is None or sorted_mat_index[i_r] != -1:
                break

        if i_r is None or sorted_mat_index[i_r] == -1:
            continue

        new_page_id[i] = sorted_page_id[i]
        new_mat_index[i] = sorted_mat_index[i_r]
        new_page_title[i] = str(sorted_page_title_ser.iloc[i])
        new_is_redirect[i] = True
        new_redirect[i] = str(sorted_page_title_ser.iloc[i_r])

In [ ]:
title_df = pd.DataFrame(
    {
        "PageID": new_page_id,
        "MatrixIndex": new_mat_index,
        "PageTitle": new_page_title,
        "IsRedirect": new_is_redirect,
        "RedirectLocation": new_redirect,
    }
)

title_df = title_df[title_df["MatrixIndex"] != -2]
title_df.head(20)

In [ ]:
title_df.to_csv(title_info_tsv_path, sep="\t", index=False)

In [ ]:
re.sub(r"^\s+|\n|\t|\||\s+$", "", "  asd dwad\n a||ds\tf  ")